**SEANCE:** 2

**NOM:** Albert Hounkpe

**PORTEUR #**: P2

**ETAPES ASSIGNEES:** 2.2 , 2.6

# Carnet de démarrage — sujet 05 · Santander Satisfaction

**Objectif du sujet** : repérer le client qui va se plaindre.

Ce carnet est un **châssis**, pas une solution. Il fait quatre choses et s'arrête :

1. il **trouve** vos fichiers, quelle que soit la plateforme ;
2. il **charge** la table efficacement — cache Parquet construit une fois, projection de colonnes ;
3. il **profile** les données : dimensions, types, cible, manquants ;
4. il **écrit** une soumission au bon format, avec une valeur de référence volontairement nulle.

**Ce qu'il ne fait pas** : choisir votre métrique, votre découpage de validation, votre modèle, ni vos
variables. Ces décisions sont le sujet, et c'est ce que la grille note.

3,96 % de mécontents. 97 colonnes sont constantes sur un échantillon.

---

## Où mettre ce carnet

| Plateforme | Ce qu'il faut faire |
|---|---|
| **Le dossier du sujet** | **(lien du Drive communiqué par l’enseignant)** — téléchargez-le avant tout (le lien figure aussi dans l'énoncé) |
| **En local (VS Code, Jupyter)** | placez le carnet **dans le dossier téléchargé**, ou définissez `DONNEES_PROJET` vers ce dossier |
| **Google Colab** | déposez les fichiers dans `MyDrive/Sujet_05_Santander_Satisfaction/` — le carnet les trouvera ; ou téléversez-les dans `/content/` |
| **Kaggle** | créez un jeu de données avec vos fichiers, **Add Input** — `/kaggle/input` est exploré automatiquement |

**Rien n'est codé en dur** : le carnet cherche, et s'il ne trouve pas, il **dit précisément** ce qu'il a
cherché et où.


## Les fichiers de ce sujet

| Fichier | Rôle |
|---|---|
| `train.csv` | entraînement ou annexe |
| `a_decider.csv` | à prédire — la cible est absente |

**Format de sortie attendu** : `ID,TARGET`.
**Colonne d'identifiant** : `ID`.
**Fichier à produire** : `soumission.csv`, à la racine de votre dépôt.


In [9]:
# --- 1. Réglages --------------------------------------------------------------
from pathlib import Path
import os
import pandas as pd

# Où télécharger vos données — le dossier de votre sujet sur le Drive du cours.
LIEN_DRIVE = "__LIEN__"

# ECHANTILLON : nombre de lignes lues pour démarrer. None = tout lire.
ECHANTILLON = 200_000
DOSSIER_CACHE = Path.cwd() / "_cache_parquet"


In [10]:
import os
os.environ["DONNEES_PROJET"] = "/kaggle/input/datasets/alberthounkpe/santander-satisfaction-files"

In [11]:
# --- 2. Trouver les données, sans rien coder en dur --------------------------
FICHIERS_ATTENDUS = ['train.csv', 'a_decider.csv']
SUJET = "Sujet_05_Santander_Satisfaction"


def trouver_donnees(fichiers=FICHIERS_ATTENDUS, sujet=SUJET):
    """Cherche le dossier qui contient TOUS les fichiers attendus."""
    essais = []
    pistes = []
    if os.environ.get("DONNEES_PROJET"):
        pistes.append(Path(os.environ["DONNEES_PROJET"]))
    ici = Path.cwd().resolve()
    pistes += [ici, ici / "donnees", ici / sujet]
    pistes += list(ici.parents)[:4]
    pistes += [Path("/kaggle/input"), Path("/content"), Path("/content/drive/MyDrive")]

    for racine in pistes:
        if not racine.exists():
            continue
        candidates = [racine, racine / sujet]
        if racine.name in ("input", "MyDrive"):
            candidates += [d for d in racine.iterdir() if d.is_dir()]
        for c in candidates:
            essais.append(str(c))
            if all((c / f).is_file() for f in fichiers):
                return c
    raise FileNotFoundError(
        "Dossier de données introuvable.\n"
        "Il faut un dossier contenant : " + ", ".join(fichiers) + "\n"
        "Solutions : placez le carnet à côté de ce dossier, "
        "ou définissez DONNEES_PROJET=/chemin/vers/le/dossier.\n"
        "Chemins essayés : " + str(essais[:12])
    )


DOSSIER = trouver_donnees()
print("dossier de données :", DOSSIER)
print("fichiers vus :", sorted(p.name for p in DOSSIER.iterdir() if p.is_file())[:12])


dossier de données : /kaggle/input/datasets/alberthounkpe/santander-satisfaction-files
fichiers vus : ['a_decider.csv', 'train.csv']


In [12]:
# --- 3. Chargement (table légère) ---------------------------------------------
def table(nom, colonnes=None, morceaux=None, complet=False):
    """Ouvre un CSV directement avec pandas — la table tient en mémoire."""
    source = DOSSIER / nom
    if not source.is_file():
        raise FileNotFoundError(f"{nom} introuvable dans {DOSSIER}")
    if morceaux:
        return pd.read_csv(source, usecols=colonnes, chunksize=morceaux)
    return pd.read_csv(source, usecols=colonnes, nrows=None if complet else ECHANTILLON)


In [13]:
# --- 4. Charger et regarder ---------------------------------------------------
entrainement = table("train.csv")
print("dimensions :", entrainement.shape)
print("colonnes :", len(entrainement.columns))
entrainement.head()


dimensions : (60816, 371)
colonnes : 371


,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
0,130807,2,26,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,117310.979016,0
1,78981,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,85472.340000,0
2,50325,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,317769.240000,0
3,146646,2,30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,76209.960000,0
4,136536,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,302754.000000,0


In [14]:
# --- 5. Profil de la cible ----------------------------------------------------
CIBLE = "TARGET"
if CIBLE in entrainement.columns:
    serie = entrainement[CIBLE]
    print("type :", serie.dtype)
    print("manquants :", int(serie.isna().sum()))
    print(serie.describe())
    if serie.nunique() <= 20:
        print(serie.value_counts(normalize=True).sort_index())
else:
    print(f"la cible {CIBLE} est absente de cette table — normal pour la table à prédire")


type : int64
manquants : 0
count    60816.000000
mean         0.039562
std          0.194929
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: TARGET, dtype: float64
TARGET
0    0.960438
1    0.039562
Name: proportion, dtype: float64


In [15]:
# --- 6. Soumission de RÉFÉRENCE (format seulement) ----------------------------
# Cette valeur n'a AUCUNE prétention prédictive : elle vérifie le format.
COLONNES_SORTIE = ['ID', 'TARGET']


def ecrire_soumission(identifiants, valeurs=None, chemin="soumission.csv"):
    """Écrit le fichier attendu. `valeurs` peut être une colonne, ou une matrice de probabilités."""
    sortie = pd.DataFrame({"ID": identifiants})
    if len(COLONNES_SORTIE) == 2:
        sortie["TARGET"] = valeurs
    else:
        n = len(COLONNES_SORTIE) - 1
        matrice = valeurs if valeurs is not None else [[1.0 / n] * n for _ in identifiants]
        for j, colonne in enumerate(COLONNES_SORTIE[1:]):
            sortie[colonne] = [ligne[j] for ligne in matrice]
    sortie.to_csv(chemin, index=False)
    print(f"écrit : {chemin} · {len(sortie):,} lignes · colonnes {list(sortie.columns)}")
    if len(COLONNES_SORTIE) > 2:
        print("somme par ligne (doit valoir 1) :", sortie.iloc[:, 1:].sum(axis=1).round(6).unique()[:3])
    return sortie


a_predire = table("a_decider.csv", colonnes=["ID"], complet=True)
identifiants = a_predire["ID"].drop_duplicates().to_numpy()
print("identifiants à prédire :", len(identifiants))
ecrire_soumission(identifiants)


identifiants à prédire : 15204
écrit : soumission.csv · 15,204 lignes · colonnes ['ID', 'TARGET']


,ID,TARGET
0,38893,None
1,133597,None
2,24908,None
3,19551,None
4,36051,None
...,...,...
15199,28686,None
15200,66259,None
15201,44895,None
15202,33938,None


In [16]:
# --- 7. Le travail commence ICI -----------------------------------------------
# À vous de décider, et de justifier :
#   - la métrique, et ce qu'elle coûte ;
#   - le découpage de validation ;
#   - les variables, leur encodage, leur imputation ;
#   - le modèle, son réglage, son seuil.
# Ce carnet ne le fera pas à votre place : c'est l'objet du projet.
raise NotImplementedError(
    "Carnet de démarrage : la modélisation est à votre charge. Écrivez votre pipeline à partir d'ici."
)


NotImplementedError: Carnet de démarrage : la modélisation est à votre charge. Écrivez votre pipeline à partir d'ici.

**###2.2 — La barre de bruit**

In [21]:
# --- 2.2 La barre de bruit (version auto-suffisante) ------------------------
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# 1. Si le découpage n'a pas été fait, on le refait ici
if 'X_val' not in globals():
    print("Variables manquantes, rechargement des données et découpage...")
    from pathlib import Path
    import pandas as pd
    
    # On cherche le dossier data
    DOSSIER = Path("/kaggle/input/santander-satisfaction-files") # Ton dossier Kaggle
    if not (DOSSIER / "train.csv").exists():
        # Si le nom du dossier change, on cherche récursivement
        import glob
        DOSSIER = Path(glob.glob("/kaggle/input/**/train.csv", recursive=True)[0]).parent
        
    train = pd.read_csv(DOSSIER / "train.csv")
    CIBLE = "TARGET"
    features = [c for c in train.columns if c not in ("ID", CIBLE)]
    X, y = train[features], train[CIBLE]
    
    # Découpage 80/20
    idx_app, idx_val = train_test_split(train.index, test_size=0.20, stratify=y, random_state=42)
    X_app, y_app = X.loc[idx_app], y.loc[idx_app]
    X_val, y_val = X.loc[idx_val], y.loc[idx_val]

# 2. Calcul du bruit
print("Calcul de la barre de bruit (Bootstrap)...")
PLANCHER_METIER = 0.6890
score_plancher_val = X_val["var15"].to_numpy()
y_val_array = y_val.to_numpy()

n_boot = 200
aucs_boot = []
rng = np.random.default_rng(42)

for _ in range(n_boot):
    idx = rng.choice(len(y_val_array), size=len(y_val_array), replace=True)
    if y_val_array[idx].sum() == 0 or y_val_array[idx].sum() == len(idx):
        continue
    aucs_boot.append(roc_auc_score(y_val_array[idx], score_plancher_val[idx]))

bruit = np.std(aucs_boot)

print(f"\nPlancher métier (AUC)     : {PLANCHER_METIER:.4f}")
print(f"Bruit de mesure (σ AUC)   : {bruit:.4f}")
print(f"Barre de bruit            : [{PLANCHER_METIER - bruit:.4f} ; {PLANCHER_METIER + bruit:.4f}]")
print(f"→ Un modèle doit dépasser {PLANCHER_METIER + bruit:.4f} pour être significativement meilleur.")

Variables manquantes, rechargement des données et découpage...
Calcul de la barre de bruit (Bootstrap)...

Plancher métier (AUC)     : 0.6890
Bruit de mesure (σ AUC)   : 0.0101
Barre de bruit            : [0.6789 ; 0.6991]
→ Un modèle doit dépasser 0.6991 pour être significativement meilleur.


**###2.6 — La comparaison**

In [29]:
# --- 2.6 La comparaison (version auto-suffisante) ---------------------------
import time
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# 1. On vérifie si les modèles ont déjà été entraînés (au cas où on relance juste cette cellule)
if 'res' not in globals():
    print("Entraînement des modèles (cela peut prendre quelques secondes)...")
    
    # On ne garde que les colonnes utiles (on enlève les constantes)
    cols = [c for c in X_app.columns if X_app[c].nunique(dropna=False) > 1]
    
    modeles = {
        "Régression logistique": make_pipeline(
            SimpleImputer(strategy="median"), 
            StandardScaler(),
            LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
        ),
        "Gradient Boosting": HistGradientBoostingClassifier(random_state=42)
    }
    
    resultats = []
    for nom, m in modeles.items():
        t = time.time()
        m.fit(X_app[cols], y_app)
        auc_app = roc_auc_score(y_app, m.predict_proba(X_app[cols])[:, 1])
        auc_val = roc_auc_score(y_val, m.predict_proba(X_val[cols])[:, 1])
        temps = round(time.time() - t, 1)
        resultats.append((nom, auc_app, auc_val, temps))
        
    # Création du tableau
    res = pd.DataFrame(resultats, columns=["modèle", "AUC app", "AUC val", "sec"])

# 2. Calcul des écarts et de la significativité par rapport au bruit
PLANCHER_METIER = 0.6890
# On utilise la variable 'bruit' calculée dans la cellule 2.2
if 'bruit' not in globals():
    bruit = 0.01 # Valeur par défaut si la cellule 2.2 n'a pas tourné

res["Écart plancher"] = (res["AUC val"] - PLANCHER_METIER).round(4)
res["Hors bruit ?"] = res["Écart plancher"].apply(lambda x: "OUI" if x > bruit else "NON")

# 3. Affichage propre
print(f"\n=== Comparaison des approches (Validation, AUC) ===")
print(f"Plancher métier : {PLANCHER_METIER:.4f} | Bruit (sigma) : {bruit:.4f}\n")

# On affiche le tableau avec un joli format
display(res[["modèle", "AUC val", "Écart plancher", "Hors bruit ?"]].style.hide(axis="index"))


=== Comparaison des approches (Validation, AUC) ===
Plancher métier : 0.6890 | Bruit (sigma) : 0.0101



modèle,AUC val,Écart plancher,Hors bruit ?
Régression logistique,0.775584,0.086600,OUI
Gradient Boosting,0.831985,0.143000,OUI
